In [ ]:
import os
import pandas as pd
from datetime import timedelta

# 设置时间间隔为10秒
time_interval = timedelta(seconds=10)

# 存储不连续时间的文件名和行号
discrepancies = []

# 获取当前工作目录
folder_path = './charge'  # charge文件夹路径
output_folder = './charge2'  # 新的文件夹路径

# 确保charge2文件夹存在
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# 遍历当前文件夹的所有excel文件
for file_name in os.listdir(folder_path):
    if file_name.endswith('.xlsx') or file_name.endswith('.xls'):
        # 读取Excel文件
        file_path = os.path.join(folder_path, file_name)
        try:
            df = pd.read_excel(file_path)

            # 假设时间在第一列
            time_column = pd.to_datetime(df.iloc[:, 0])

            # 创建一个新DataFrame来存放修复后的数据
            new_df = df.copy()

            # 记录插值的索引和新插入的时间
            insert_rows = []

            # 检查时间是否连续并插入缺失的时间行
            for i in range(1, len(time_column)):
                # 计算当前行和前一行的时间差
                time_diff = time_column[i] - time_column[i-1]
                
                # 检查是否不连续
                if time_diff != time_interval:
                    # 记录不连续信息
                    discrepancies.append({
                        '文件名': file_name,
                        '不连续行号': i + 1,  # 行号从1开始
                        '前一个时间': time_column[i-1],
                        '当前时间': time_column[i]
                    })

                    # 计算应该插入多少行
                    num_missing_rows = int(time_diff.total_seconds() / 10) - 1

                    # 插入线性插值
                    for j in range(1, num_missing_rows + 1):
                        missing_time = time_column[i-1] + j * time_interval
                        
                        # 线性插值填充其他列的数据
                        new_row = df.iloc[i-1].copy()  # 从前一行拷贝数据
                        new_row[0] = missing_time  # 插入缺失的时间
                        
                        # 对其他列进行线性插值（使用pandas的插值功能）
                        for col in range(1, df.shape[1]):
                            # 插值其他列的值
                            new_row[col] = df.iloc[i-1: i+1, col].interpolate().iloc[1]
                        
                        # 插入新行数据
                        insert_rows.append((i + len(insert_rows), new_row))

            # 将插值行插入到原始数据中
            for index, row in insert_rows:
                new_df = pd.concat([new_df.iloc[:index], pd.DataFrame([row]), new_df.iloc[index:]]).reset_index(drop=True)

            # 将修复后的数据保存回新的文件夹
            new_file_name = os.path.join(output_folder, f"fixed_{file_name}")
            new_df.to_excel(new_file_name, index=False)

        except Exception as e:
            print(f"读取文件 {file_name} 时出错: {e}")

# 将不连续记录保存到新的Excel文件中
if discrepancies:
    discrepancies_df = pd.DataFrame(discrepancies)
    discrepancies_df.to_excel(os.path.join(output_folder, 'time_discrepancies.xlsx'), index=False)
    print("不连续的时间已记录到 'charge/time_discrepancies.xlsx'")
else:
    print("未发现时间不连续的情况")
